In [0]:
CATALOG = "energy_puls"
LANDING = f"/Volumes/{CATALOG}/landing/landing"
BRONZE = f"{CATALOG}.bronze"
CKPT = f"/Volumes/{CATALOG}/landing/_checkpoints"
from pyspark.sql import functions as F
print("ready")

In [0]:
def run_dpe():
    return (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", f"{CKPT}/dpe_schema")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .load(f"{LANDING}/dpe/weekly/")
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_extract_week",
                    F.regexp_extract("_source_file", r"(\d{4}-W\d{2})", 1))
        .writeStream
        .option("checkpointLocation", f"{CKPT}/dpe")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{BRONZE}.dpe_raw"))

try:
    run_dpe().awaitTermination()
except Exception as e:
    if "UnknownFieldException" in str(type(e)) or "new columns" in str(e):
        print("Schema evolved — restarting once")
        run_dpe().awaitTermination()
    else:
        raise
print("dpe → Bronze done")

In [0]:
dq = spark.sql(f"""
  SELECT _extract_week,
         COUNT(*)                                             AS rows,
         COUNT(*) - COUNT(DISTINCT numero_dpe)                AS dup_or_rediag,
         SUM(CASE WHEN surface_habitable < 0 THEN 1 END)      AS neg_surface,
         SUM(CASE WHEN code_commune NOT RLIKE '^[0-9]{{5}}$'
                  THEN 1 END)                                 AS bad_commune,
         SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 END)  AS rescued
  FROM {BRONZE}.dpe_raw
  GROUP BY 1 ORDER BY 1
""")
dq.write.mode("overwrite").saveAsTable(f"{BRONZE}.dpe_dq_metrics")
display(dq)

In [0]:
display(spark.read.option("header","true").csv(f"{LANDING}/dpe/weekly/2023-W01/dpe_extract.csv").limit(20))

In [0]:
display(spark.sql(f"SELECT * FROM {BRONZE}.dpe_raw LIMIT 20"))